# 03 · Genie Agent: questions over governed tables and SOP documents (executed)

The app calls Genie in **Agent mode** (`/api/2.0/genie/agents/{space}/responses`), which can query the
21 governed tables **and** read the 6 SOP/recipe PDFs in the Unity Catalog volume. These are live answers.

In [1]:
from proof_helpers import *
print('catalog.schema:', FQ)

catalog.schema: fevm_cme_conde_catalog.kaapi_bricks


In [2]:
questions = [
    "Which 5 store-ingredient positions are most urgently below reorder threshold? Show store, ingredient, stock, threshold and days of cover.",
    "Which supplier has the lowest on-time delivery rate, and what is it?",
    "How long can prepared filter coffee decoction be kept before it must be discarded?",
    "What are the allergen handling steps when a customer reports a nut allergy?",
]
for q in questions:
    r = genie_ask(q)
    print("Q:", q)
    print(f"   status={r['status']}  {r['seconds']} s  tools used={r['tools']}")
    for s in r["sql"][:1]:
        print("   SQL:", " ".join(s.split())[:400])
    print("A:", r["answer"][:900].replace("\n", "\n   "))
    print("-" * 100)

Q: Which 5 store-ingredient positions are most urgently below reorder threshold? Show store, ingredient, stock, threshold and days of cover.
   status=completed  13.5 s  tools used=['execute_sql']
   SQL: SELECT store_id, ingredient_name, current_stock, reorder_threshold, days_cover FROM `fevm_cme_conde_catalog`.`kaapi_bricks`.`gold_inventory_position` WHERE below_reorder = true AND days_cover IS NOT NULL ORDER BY days_cover ASC LIMIT 5
A: ## Most Urgent Below-Reorder Store-Ingredient Positions
   
   Ranked by lowest days of inventory cover remaining:
   
   **Top 5 urgent below-reorder positions**
   
   | store_id | ingredient_name | current_stock | reorder_threshold | days_cover |
   | --- | --- | --- | --- | --- |
   | STR-010 | Cocoa Powder | 0.38 | 4.0 | 1.4 |
   | STR-031 | Jaggery | 2.86 | 20.0 | 3.5 |
   | STR-011 | Badam Paste | 3.89 | 5.0 | 13.1 |
   | STR-011 | Nannari Syrup | 3.81 | 6.0 | 13.1 |
   | STR-016 | Jaggery | 16.45 | 20.0 | 13.4 |
   
   **STR-010's Cocoa Powde

Q: Which supplier has the lowest on-time delivery rate, and what is it?
   status=completed  12.5 s  tools used=['execute_sql']
   SQL: SELECT supplier_id, supplier_name, category, on_time_rate, total_pos, delivered_pos FROM `fevm_cme_conde_catalog`.`kaapi_bricks`.`gold_supplier_performance` WHERE on_time_rate IS NOT NULL ORDER BY on_time_rate ASC LIMIT 5
A: ## Lowest On-Time Delivery Rate
   
   **Organic Alternatives India** (SUP-008, Alt Milks category) has the lowest on-time delivery rate at **80.49%**, having delivered on time for 225 of 246 purchase orders. This is the lowest among all suppliers, slightly below Kerala Spice Traders (82.9%) and Chennai Packaging Co. (83.1%).
   
   **Lowest on-time delivery rate suppliers**
   
   | supplier_id | supplier_name | category | on_time_rate | total_pos | delivered_pos |
   | --- | --- | --- | --- | --- | --- |
   | SUP-008 | Organic Alternatives India | Alt Milks | 0.8049 | 246 | 225 |
   | SUP-005 | Kerala Spice Traders | Spices & Fla

Q: How long can prepared filter coffee decoction be kept before it must be discarded?
   status=completed  13.5 s  tools used=['search_context_files']
A: ## Filter Coffee Decoction Hold Time
   
   Prepared filter coffee decoction must be discarded after a **maximum of 4 hours**. This is a firm limit — after 4 hours the decoction oxidizes, turns bitter, and loses its aroma, so it must be discarded regardless of how it looks or smells.
   
   This standard is consistent across the Food Safety Policy and the Drink Recipes SOP, and is reinforced in closing procedures: at the end of each shift, baristas must discard all remaining prepared decoction rather than carry it over to the next day.
----------------------------------------------------------------------------------------------------


Q: What are the allergen handling steps when a customer reports a nut allergy?
   status=completed  15.4 s  tools used=['search_context_files']
A: ## Nut Allergy Handling Procedure
   
   Per the Kaapi Bricks Food Safety Policy, nut allergens in the menu include almond milk, badam milk (PRD-020), pistachio in Pista Kaapi (PRD-026), and hazelnut syrup. When a customer reports a nut allergy, staff follow the **Allergen Communication Protocol**:
   
   1. **Confirm the specific allergen** — verify it is a tree nut allergy and which nut(s) specifically.
   2. **Check all ingredients, including syrups and add-ons** — review the full recipe (e.g., hazelnut syrup, badam/pistachio paste, almond milk) for hidden nut content.
   3. **Use freshly cleaned equipment to prevent cross-contact** — the blender must be cleaned between orders when switching between nut-based and dairy milks, and almond milk/coconut milk must be stored and dispensed separately from dairy milk.
   4. **If unsure, consult t